# Featurization: How Do You describe a Material to a Machine?

# 🧪💻 

# Lecture recording
    
:::{seealso} Please refer to the lecture recording on Canvas 
:::

<img src="images/lec05-Chem_formulas.png" width="500">



## The main idea 💡

**Featurization means turning a material into a list of numbers that a computer can work with - while still keeping the important chemistry that makes that material special!**

Everything follows from this basic idea.

---


## 1. The problem: models are good with arithmetic, but chemistry is written in symbols
🔢↔️🔡 

Computers talk in the [language of numbers](week00-1_lecture.ipynb) - written in the binary number system $(0,1)$. So, when we are processing our data using machine learning models like linear regression or k-means clustering, computers work well in finding patterns in the data because they speak the language of numbers. So the data input should always be in exactly one shape:

> a **fixed-length vector of real numbers**, `x = [x₁, x₂, ..., xₙ]`, the same length for every sample.

But materials data almost always come with chemical formulas that are not in number format:

```
SiO₂        GaAs        Ca₃(PO₄)₂        Fe–18Cr–8Ni        Li₇La₃Zr₂O₁₂
```

To you, these formulas carry a lot of meaning. You look at `GaAs` and think: III–V semiconductor, direct band gap, used in LEDs and solar cells. You look at `Fe–18Cr–8Ni` and think austenitic stainless, corrosion resistance from the chromium.

But to a computer, `"SiO2"` is just four random characters, and is no more different to "J1Pq". There is no meaning.😕 

So, we need to turn the chemical formulas into a number format, but how?

The easiest move is to give each formula an ID number: `SiO₂ = 1`, `GaAs = 2`, `Al₂O₃ = 3`. 

But this is problematic,

**Problem 1 - it invents an order that doesn't exist.** The model now believes `Al₂O₃` is somehow *greater than* `GaAs`, and that `GaAs` sits *between* `SiO₂` and `Al₂O₃`. Pure nonsense, but the model doesn't know that.

**Problem 2 - even after you fix that, everything is equally unrelated.** If you plot the material number on a graph, now *every material sits exactly the same distance from every other material.*  



<img src="images/lec05_fake_rep.png" width="500">



**💡** It's not just about *what* data is in the set, **How** data is represented is just as important!

A representation is not merely a storage format. *A representation decides which things count as similar.* If you choose it badly, you would have told the model incorrect information before training begins.

<img src="images/lec05_fig1.svg" width="500">
---

## 2. What is featurization

Featurization (also *descriptor generation*, or *encoding*) is simply a way to mathematically represent the chemistry behind the formula:


On in other words, when your dataset contains chemical formulas - you convert them into a numerical format using a certain function.

### What makes a good featurization

| Requirement | Why it matters |
|---|---|
| **Numerical and fixed-length** | Algorithms need a constant input size. But formulas contain varying numbers of elements - this is the central technical obstacle. So, formulas with different lengths need to be converted into a fixed-length numerical vector.|
| **Invariant to what shouldn't matter** | `SiO₂` and `O₂Si` must map to the *same* vector. So should `SiO₂` and `Si₂O₄`, since they describe the same compound. |
| **Preserves meaningful similarity** | Chemically similar materials should land near each other in vector space. |
| **Discriminative** | Materials with genuinely different properties must not collapse onto the same vector. |
| **Cheap** | Computable from information you actually have for every entry in your dataset. |
| **Interpretable** *(nice to have)* | Lets you ask afterwards: *which feature drove this prediction?* |

Notice that requirements 2 and 4 are somewhat against each other. Every featurization deliberately discards information; so we have to be careful in discarding the *right* information.

### An analogy 💁

A person's name is a perfect unique identifier and completely useless for prediction. If you wanted a model to predict who is going to do a job well, you would not feed it names - instead you'd feed their years of experience, education level, prior roles, test scores. Those are useful features.

Who is likely to do a better job at building a machine learning model?
| Name |
|---|
|John Anderson|
|Claire Smith|

That doesn't help, does it? although we have the names of the people, they don't say anything about their abilities.
But if ignore the names and look at the things associated with the names:

| Name | Previous Role | Qualifications |
|---|---|---|
|John Anderson| Marketing Executive | Masters in Creative Arts |
|Claire Smith| Data Scientist | Masters in Computer Science |

Now we can make a better decision.


Similarly `SiO₂` is only a name, it doesn't tell us what it can do! So we need to look past the name and tell the model what that material is capable of.

---


## 3. The ladder of materials representations


🪜 Materials data arrives at several levels of detail, and each level has its own family of representations.

| What you know about the material | Typical representation | Cost / availability |
|---|---|---|
| **Composition only** (the formula) | Composition-based feature vectors - **this lecture** | Cheapest. Available for *everything*. |
| Composition **+ crystal structure** | Structural descriptors, Voronoi-based features, crystal graph neural networks | Needs a solved structure or a DFT calculation |
| **Microstructure** (images) | Convolutional neural network features | Needs imaging - *this is the defect-classification route* |
| **Literature** (text) | Word embeddings trained on papers | Needs a large text corpus |

Composition-based featurization is the bottom rung, and that is exactly why it is the right place to start. **The chemical formula is the one piece of information present for every material in every dataset ever assembled.** It also works considerably better than you would think - which is rather interesting to have a look at.

---


## 4. Where does the chemistry hide inside a formula?


A chemical formula carries exactly two pieces of information:

1. **Which elements** are present - *identity*
2. **How many of each** - *stoichiometry*

Stoichiometry is already numbers; that part is easy. Identity is where all the richness sits, because **an element symbol is a pointer.**

`Si` is a reference to a specific row of the periodic table, and behind that reference sit dozens of measured and computed numbers:

> atomic mass · electronegativity · atomic and covalent radius · melting point · boiling point · electron affinity · ionization energy · number of valence s, p, d, f electrons · group and row · ground-state band gap · molar volume · bulk modulus · thermal conductivity · Mendeleev number · space group of the pure element …

**Featurization step one is simply: follow the pointer and copy the numbers in.**

<img src="images/lec05_fig2.svg" width="500">

A small slice of such a table:

| | mass (u) | electroneg. | m.p. (°C) | b.p. (°C) | el. affinity (kJ/mol) | 1st ionization (kJ/mol) | cov. radius (pm) | bulk mod. (GPa) | therm. cond. (W/m·K) |
|---|---|---|---|---|---|---|---|---|---|
| **Si** | 28.085 | 1.90 | 1414 | 2900 | 133.6 | 786.5 | 111 | 100 | 150 |
| **Ti** | 47.867 | 1.54 | 1668 | 3287 | 7.6 | 658.8 | 176 | 110 | 22 |

*Values are illustrative and rounded, as in the lecture slides; consult a reference table such as [ptable.com](https://ptable.com) for exact figures.*

If you read the row for silicon as a single object you will have:

```
Si = [28.085, 1.90, 1414, 2900, 133.6, 786.5, 111, 100, 150]
```
💡

There are 9 data points here. So, if you plot these 9 data points in a 9-dimensional graph (of course we can't imagine it, but computers don't have any problem with doing it mathematically) then Silicon is now a **single point in a 9-dimensional property space.** 

If you do the same thing with other elements: in that space, Si will sit close to Ge and far from Na - not because anyone programmed the periodic table in, but because *their numbers genuinely are alike.* Chemical intuition has been converted into geometry, and geometry is something a machine can compute with.
🤯

---

## 5. From elements to compounds - the aggregation problem

We now have numerical vectors for *elements*. But materials are not elements they are *compounds*. We need to combine element vectors into one vector describing the whole material, and three problems get in the way.

**Problem A - stoichiometry.** `SiO₂` is not "Si plus O". There are two oxygens for every silicon, so oxygen's contribution should count twice as heavily.

**Problem B - scale.** `SiO₂` has 3 atoms per formula unit, `Si₂O₄` has 6, `Si₂O₃` has 5. Raw atom counts make chemically identical descriptions look different, and let compounds with big formula units dominate on size alone.

*A and B can be solved in one stroke by using **fractional composition**:*

```
SiO₂  →  Si: 1/3 = 0.333    O: 2/3 = 0.667
```

Fractions always sum to 1, which normalizes away formula-unit size while preserving the ratio. `SiO₂` and `Si₂O₄` now map identically - as they should.

**Problem C - the number of elements varies.** `SiO₂` has two elements. A high-entropy alloy has five or six. Stacking element vectors end to end would give a different vector length for every material, and the model requires one fixed length.

So,

> **Don't concatenate (stack) the element vectors. Summarize them.**

Take one element property at a time - say electronegativity - and compute *statistics* across the elements present, weighted by their fractions:

| Statistic | What it tells the model |
|---|---|
| weighted **mean** | the typical value for this material |
| weighted **spread** (variance or mean absolute deviation) | how chemically *mixed* this material is |
| **min** and **max** | the extremes present |
| **range** = max − min | the internal contrast |
| value for the **majority element** | what dominates by amount |

### Why the mean alone is not enough

Consider two hypothetical compounds whose constituent electronegativities average out to the same number: one built from two moderately electronegative elements, the other from one very high and one very low. Identical means - but the second has a large electronegativity *difference*, and that difference is precisely what makes a bond ionic rather than covalent.

But the mean is the same for both. However, **Range and spread** differentiates the two. Every statistic beyond the mean exists to recover some kind of internal contrast that averaging destroyed.

### The payoff

If you have **P** element properties and compute **S** statistics for each, you always get **P × S** features.

Two elements or seven. Simple oxide or high-entropy alloy. **Always the same length.** Problem C is also fixed!.

Note:This is not a universal step in featurization - it's the standard solution to one specific problem. And you might see this in some common featurizers such as `magpie`.

<img src="images/lec05_fig3.svg" width="500">

---


## 6. Different types of composition-based featurization

Given that recipe, the remaining question is *which element properties go into the table?* There are three different ideas on this.

### A. Hand-built physical property tables

Someone with chemical expertise puts together a list of element properties they judge as relevant, drawing on measured data and DFT calculations. **Magpie** (Ward et al.), **Oliynyk**, and **JARVIS** are the widely used examples. Domain knowledge is captured through human judgement about what belongs on the list.

*Strength:* interpretable, physically grounded, works with small datasets.
*Weakness:* what matters for each ML problem might be different.

### B. Learned embeddings (Deep Learning)

Humans don't decide the properties; a vector for each element is *discovered* from data by the algorithm.

- **[mat2vec](https://www.nature.com/articles/s41586-019-1335-8)** learns element vectors from the text of materials-science papers, using the same machinery as word embeddings in natural language processing. Elements appearing in similar textual contexts end up with similar vectors.
- **[Atom2Vec](https://arxiv.org/pdf/1807.05617)** learns from which elements co-occur in known compounds in a materials database - the chemical analogue of "you shall know a word by the company it keeps."

The striking result from both lines of work is that **you can re-discover the structure of the periodic table from the data** without ever being supplied. 

### C. One-hot / fractional encoding - the control experiment

Here you list all the elements of the periodic table as column headings, and then for each compound, you fill in the presence of these elements. So it's a vector as long as the periodic table, carrying the presence (or absence) of an element in each slot. 

Or you can turn this into a fractional encoding by considering the fraction of each element in its slot and zeros everywhere else:

| | … | O | … | Si | … |
|---|---|---|---|---|---|
| **SiO₂** (one-hot) | 0 | 1 | 0 | 1 | 0 |
| **SiO₂** (fractional) | 0 | 0.667 | 0 | 0.333 | 0 |

This contains identity and stoichiometry and **no chemistry whatsoever.** Si and Ge are back to being as unrelated as Si and uranium.

And that's exactly why you should test against it as a baseline.

If your fancy, physics-based featurizer can't beat plain one-hot encoding on your data, something's wrong - either with your featurizer, or with your data. Either way, that's useful to know!

This isn't just a nice theory. [Murdock, Kauwe, Wang, and Sparks actually tested this](https://chemrxiv.org/doi/full/10.26434/chemrxiv.11879193.v1). They found that with lots of data, even simple encodings - including totally random noise - can perform just as well as fancy, physics-based descriptors. But when data is small or not very representative, the physics-based descriptors win. 

Here's the catch: most real materials datasets are small and not fully representative. So this is good news for physics-based featurization.

The CBFV package makes this easy. It bundles several featurizers (jarvis, magpie, mat2vec, oliynyk, onehot, and a random_200 control) behind one simple interface. Switching between them takes just one word - so running the "random noise" test costs you almost nothing. 🎲



## 7. What composition-based featurization cannot see

Being clear about the limits is not a disclaimer; it is where the next research question comes from.

**Polymorphs are completely invisible.** Diamond and graphite are both `C`. Anatase, rutile and brookite are all `TiO₂`. Same formula → identical feature vector → the model is *forced* to output a single prediction for materials with drastically different properties. There is no amount of training data that fixes this.

<img src="images/lec05_fig4.svg" width="500">

**Processing history and microstructure are invisible.** Two steel samples of identical composition, one quenched and one annealed, differ enormously in hardness. Composition alone cannot distinguish them - which is exactly why the image-based approach exists as a separate rung on the ladder.

**The element property tables have holes.** Values are missing for rare, radioactive and synthetic elements. And there is a subtler issue worth considering: the melting point of *pure elemental* silicon is not obviously the right number to describe silicon's behaviour *inside* a compound. These descriptors are only proxies (close values).

**The features are heavily correlated.** Twenty statistics across a hundred properties produces a great deal of redundancy. E.g. 'atomic number' and 'atomic weight' may be used as two features in the vector - but they are correlated and points to the same thing.

**Extrapolation is not free.** A model trained on oxides has learned nothing about nitrides. The vector space is continuous and invites interpolation everywhere - but your training data occupies only a small, oddly shaped region of it, and the model won't warn you when you've left.

---

## References


**Composition-based featurization and comparisons**

- Ward, L., Agrawal, A., Choudhary, A. & Wolverton, C. *A general-purpose machine learning framework for predicting properties of inorganic materials.* npj Computational Materials **2**, 16028 (2016). [doi:10.1038/npjcompumats.2016.28](https://doi.org/10.1038/npjcompumats.2016.28) — the Magpie attribute set. Open access.
- Murdock, R. J., Kauwe, S. K., Wang, A. Y.-T. & Sparks, T. D. *Is domain knowledge necessary for machine learning materials properties?* Integrating Materials and Manufacturing Innovation **9**(3), 221–227 (2020). [doi:10.1007/s40192-020-00179-z](https://doi.org/10.1007/s40192-020-00179-z) — the featurizer comparison behind §6C.
- Oliynyk, A. O. *et al.* *High-throughput machine-learning-driven synthesis of full-Heusler compounds.* Chemistry of Materials **28**(20), 7324–7331 (2016). [doi:10.1021/acs.chemmater.6b02724](https://doi.org/10.1021/acs.chemmater.6b02724) — source of the Oliynyk property set.

**Learned element embeddings**

- Tshitoyan, V. *et al.* *Unsupervised word embeddings capture latent knowledge from materials science literature.* Nature **571**, 95–98 (2019). [doi:10.1038/s41586-019-1335-8](https://doi.org/10.1038/s41586-019-1335-8) — mat2vec.
- Zhou, Q., Tang, P., Liu, S., Pan, J., Yan, Q. & Zhang, S.-C. *Learning atoms for materials discovery.* PNAS **115**(28), E6411–E6417 (2018). [doi:10.1073/pnas.1801181115](https://doi.org/10.1073/pnas.1801181115) — Atom2Vec. Preprint: [arXiv:1807.05617](https://arxiv.org/abs/1807.05617).

**Databases and software**

- Choudhary, K. *et al.* *The joint automated repository for various integrated simulations (JARVIS) for data-driven materials design.* npj Computational Materials **6**, 173 (2020). [doi:10.1038/s41524-020-00440-1](https://doi.org/10.1038/s41524-020-00440-1) · [jarvis.nist.gov](https://jarvis.nist.gov)
- CBFV package — [github.com/Kaaiian/CBFV](https://github.com/Kaaiian/CBFV)
- Materials Project — [materialsproject.org](https://next-gen.materialsproject.org/)
- Element property reference — [ptable.com](https://ptable.com)

---



